# 03 — Development Finance LGD Model

**Australian Bank Practice — Scenario-Based Approach**

Development finance is the most complex and cyclically sensitive lending product.
This notebook demonstrates:

| Step | Description |
|------|-------------|
| 1 | Load synthetic development finance default data |
| 2 | EDA — LGD by completion stage, development type, pre-sales |
| 3 | Completion-stage segmentation (THE primary LGD driver) |
| 4 | Fund-to-complete vs sell-as-is decision analysis |
| 5 | Cost-to-complete as dominant cost |
| 6 | Scenario analysis — GRV decline, cost overrun, pre-sale rescission |
| 7 | Downturn overlay by completion stage |
| 8 | MoC (highest of all products: +5pp) |
| 9 | APRA slotting framework |
| 10 | Validation |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.data_generation import generate_development_data
from src.lgd_calculation import DevelopmentLGDEngine, exposure_weighted_average
from src.validation import (
    accuracy_metrics, conservatism_test, calibration_by_segment,
    compute_psi, generate_validation_report
)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.join('..', 'outputs')
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)

## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy (standard policy):** use observed workout inputs first, then approved proxy inputs, then conservative fallback with explicit disclosure.
- **Proxy logic (standard policy):** proxy arrears and behavioural drivers are allowed for portfolio demonstration where observed collections history is unavailable.
- **Cure treatment (standard policy):** development uses a simplified cure overlay proxy only; it is not forced into all products.
- **Calibration status (standard policy):** this notebook is a transparent proxy baseline and is **not** internally calibrated to real workout tapes.
- **Product differentiation:** development severity is driven by GRV risk, completion %, cost-to-complete, and scenario path (fund-to-complete vs sell-as-is), not mortgage LMI or operating-cashflow-only signals.
- **Output standard:** report `lgd_base`, `lgd_downturn`, and EAD-weighted outputs for interview-ready comparison.


## Objective
Build an interview-ready development finance LGD module with stage-aware recovery logic and weighted outputs.

## Drivers
- GRV decline
- Completion stage and completion percentage
- Cost-to-complete and sell-through speed
- Exit path (fund-to-complete vs sell-as-is)

## Logic
- Scenario-based development severity engine with product-appropriate discounting and weighted LGD

## Downturn
- Macro-linked stress via deeper GRV haircuts, cost overruns, and slower sell-through

## Validation
- EAD-weighted base/downturn outputs, slotting checks, and governance flags

## Limitations
- Synthetic portfolio and proxy assumptions; future calibration requires internal workout data


## 1. Generate & Load Data

In [ ]:
loans, cashflows = generate_development_data(n_loans=200, seed=44)

print(f"Loans: {loans.shape}")
print(f"Cashflows: {cashflows.shape}")
print(f"\nCompletion stage distribution:")
display(loans['completion_stage'].value_counts())
print(f"\nFund-to-complete rate: {loans['fund_to_complete'].mean():.1%}")
loans.head()

## 2. Exploratory Data Analysis

In [ ]:
print("=== Realised LGD Summary ===")
display(loans['realised_lgd'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ewa_realised = exposure_weighted_average(loans, 'realised_lgd', 'ead')
axes[0].hist(loans['realised_lgd'], bins=40, edgecolor='white', alpha=0.8, color='darkorange')
axes[0].axvline(ewa_realised, color='red', ls='--',
                label=f"EAD-weighted: {ewa_realised:.2%}")
axes[0].set_title('Development Finance ? Realised LGD Distribution')
axes[0].set_xlabel('LGD')
axes[0].legend()

# Completion stage is THE key driver
stage_order = ['Pre-Construction', 'Early Construction', 'Mid-Construction', 'Near-Complete', 'Complete Unsold']
loans['completion_stage'] = pd.Categorical(loans['completion_stage'], categories=stage_order, ordered=True)
loans.boxplot(column='realised_lgd', by='completion_stage', ax=axes[1], rot=20)
axes[1].set_title('Realised LGD by Completion Stage')
axes[1].set_xlabel('Completion Stage at Default')
axes[1].set_ylabel('Realised LGD')
plt.suptitle('')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'development_lgd_overview.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Key driver analysis
print('=== Exposure-Weighted LGD by Key Segments ===')
for col in ['completion_stage', 'development_type', 'default_trigger', 'fund_to_complete']:
    print()
    print(f"--- {col} ---")
    rows = []
    for seg, grp in loans.groupby(col, observed=True):
        rows.append({
            col: seg,
            'count': len(grp),
            'total_ead': grp['ead'].sum(),
            'ead_weighted_lgd': exposure_weighted_average(grp, 'realised_lgd', 'ead'),
            'median_lgd': grp['realised_lgd'].median(),
            'mean_completion': grp['completion_pct'].mean(),
        })
    summary = pd.DataFrame(rows).set_index(col).sort_values('ead_weighted_lgd', ascending=False).round(4)
    display(summary)


In [ ]:
# Completion % vs LGD — coloured by fund-to-complete decision
fig, ax = plt.subplots(figsize=(10, 6))
for ftc, colour, label in [(1, 'green', 'Fund to Complete'), (0, 'salmon', 'Sell As-Is')]:
    mask = loans['fund_to_complete'] == ftc
    ax.scatter(loans.loc[mask, 'completion_pct'], loans.loc[mask, 'realised_lgd'],
              alpha=0.5, s=25, color=colour, label=label)
ax.set_xlabel('Completion % at Default')
ax.set_ylabel('Realised LGD')
ax.set_title('Completion Stage vs LGD — Fund-to-Complete Decision')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'development_completion_vs_lgd.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Completion-Stage Segmentation

**Completion stage at default is the single most important LGD driver** in development finance.
```
Level 1: Completion stage (5 bands)
  Level 2: Development type
    Level 3: Pre-sale coverage band
```

In [ ]:
engine = DevelopmentLGDEngine(moc=0.05)

# Primary segmentation
lr_lgd = engine.compute_long_run_lgd(loans, segments=['completion_stage'])
print("=== Exposure-Weighted Long-Run LGD by Completion Stage ===")
display(lr_lgd)

# With pre-sale band
segmented = engine.segment_loans(loans)
lr_lgd_detail = exposure_weighted_average(
    segmented, 'realised_lgd', 'ead',
    group_col=['completion_stage', 'presale_band']
)
print("\n=== Completion Stage × Pre-sale Band ===")
display(lr_lgd_detail)

## 4. Fund-to-Complete vs Sell-As-Is

In [ ]:
# Compare outcomes: fund-to-complete vs sell-as-is
rows = []
for (stage, ftc), grp in loans.groupby(['completion_stage', 'fund_to_complete'], observed=True):
    rows.append({
        'completion_stage': stage,
        'fund_to_complete': ftc,
        'count': len(grp),
        'total_ead': grp['ead'].sum(),
        'ead_weighted_lgd': exposure_weighted_average(grp, 'realised_lgd', 'ead'),
        'mean_ctc': grp['cost_to_complete'].mean(),
    })
ftc_analysis = pd.DataFrame(rows).set_index(['completion_stage', 'fund_to_complete']).round(4)

print("=== Fund-to-Complete Analysis ===")
print("(fund_to_complete: 0=Sell As-Is, 1=Fund to Complete)")
display(ftc_analysis)


## 5. Cost-to-Complete Analysis

In [ ]:
# Cost breakdown from cashflows
cost_cfs = cashflows[cashflows['cashflow_category'] == 'Cost']
cost_breakdown = cost_cfs.groupby('cashflow_type').agg(
    count=('amount', 'size'),
    total_amount=('amount', 'sum'),
    total_pv=('amount_pv', 'sum'),
).sort_values('total_pv', ascending=False)

cost_breakdown['pct_of_total'] = cost_breakdown['total_pv'] / cost_breakdown['total_pv'].sum()

print("=== Cost Breakdown (PV) ===")
display(cost_breakdown.round(2))

# Visualise
fig, ax = plt.subplots(figsize=(10, 5))
cost_breakdown['total_pv'].plot.barh(ax=ax, color='coral', edgecolor='white')
ax.set_xlabel('Total PV of Costs ($)')
ax.set_title('Development Finance — Cost Composition')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'development_cost_breakdown.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n→ Cost-to-complete dominates total workout costs in development finance.")

## 6. Scenario Analysis

Test LGD sensitivity to key stress parameters:
- **GRV decline**: property market downturn
- **Cost overrun**: construction cost inflation
- **Pre-sale rescission**: sunset clause exercise
- **Sales extension**: longer time to sell in weak market

In [ ]:
# Define scenarios
scenarios = {
    'Base Case': dict(grv_decline=0.0, cost_overrun=0.0, rescission_rate=0.0, sales_extension_months=0),
    'Mild Stress': dict(grv_decline=-0.10, cost_overrun=0.05, rescission_rate=0.10, sales_extension_months=3),
    'Moderate Downturn': dict(grv_decline=-0.20, cost_overrun=0.10, rescission_rate=0.20, sales_extension_months=6),
    'Severe Downturn': dict(grv_decline=-0.30, cost_overrun=0.15, rescission_rate=0.30, sales_extension_months=9),
    'Extreme Stress': dict(grv_decline=-0.40, cost_overrun=0.20, rescission_rate=0.40, sales_extension_months=12),
}

scenario_results = []
for name, params in scenarios.items():
    s = engine.scenario_analysis(loans, **params)
    s_eval = s.assign(realised_lgd=s['scenario_lgd'])
    ewa_lgd = exposure_weighted_average(s_eval, 'realised_lgd', 'ead')
    scenario_results.append({
        'Scenario': name,
        'EAD-Weighted LGD': ewa_lgd,
        'P95 LGD': s['scenario_lgd'].quantile(0.95),
        **params,
    })

scenario_df = pd.DataFrame(scenario_results)
print("=== Scenario Analysis Results ===")
display(scenario_df.round(4))


In [ ]:
# Visualise scenario impact
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c', '#8e44ad']
bars = ax.bar(scenario_df['Scenario'], scenario_df['EWA LGD'], color=colors, edgecolor='white')
for bar, val in zip(bars, scenario_df['EWA LGD']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.1%}', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Exposure-Weighted LGD')
ax.set_title('Development Finance — Scenario Analysis')
ax.set_ylim(0, scenario_df['EWA LGD'].max() * 1.25)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'development_scenario_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7–8. Regulatory Pipeline: Downturn + MoC

In [ ]:
result = engine.apply_overlays(loans)

pipeline_cols = ['lgd_base', 'lgd_downturn', 'lgd_with_moc', 'lgd_final']

print('=== Regulatory Pipeline ===')
display(result[pipeline_cols].describe().round(4))

print()
print('=== Exposure-Weighted by Completion Stage ===')
for stage in stage_order:
    sub = result[result['completion_stage'] == stage]
    if len(sub) == 0:
        continue
    ewa = exposure_weighted_average(sub, 'lgd_final')
    s_min = sub['downturn_scalar'].min()
    s_max = sub['downturn_scalar'].max()
    print(f"  {stage:25s}: Final LGD={ewa:.2%}  (scalar range={s_min:.2f}-{s_max:.2f}, +MoC {engine.moc:.0%})  n={len(sub)}")


## Cure Treatment (Pre-Step-3 Baseline)

For development exposures, cure is represented as a simplified proxy overlay applied to liquidation-path LGD.

- Proxy drivers: completion stage, presale coverage, and LVR as-if-complete.
- Overlay is capped under policy settings and disclosed as proxy treatment.

Policy position: fallback and proxy rules follow `docs/fallback_hierarchy.md`; calibration remains proxy-only pending internal data.


In [ ]:
# Simplified cure overlay for development loans
stage_cure_base = {
    'Pre-Construction': 0.02,
    'Early Construction': 0.04,
    'Mid-Construction': 0.07,
    'Near-Complete': 0.12,
    'Complete Unsold': 0.10,
}

result['cure_rate_proxy'] = result['completion_stage'].map(stage_cure_base).fillna(0.05)
result['cure_rate_proxy'] += 0.12 * (result['presale_coverage'] - 0.50).clip(lower=0, upper=0.50)
result['cure_rate_proxy'] -= 0.15 * (result['lvr_as_if_complete'] - 0.70).clip(lower=0, upper=0.40)
result['cure_rate_proxy'] = result['cure_rate_proxy'].clip(0.01, 0.25)

result['liquidation_loss_proxy'] = result['lgd_final']
result['lgd_after_cure_proxy'] = ((1 - result['cure_rate_proxy']) * result['liquidation_loss_proxy']).clip(0, 1)

print('=== Simplified Cure Overlay Impact (Development) ===')
print(f"EAD-weighted cure rate proxy: {exposure_weighted_average(result, 'cure_rate_proxy', 'ead'):.2%}")
print(f"EAD-weighted liquidation LGD: {exposure_weighted_average(result, 'liquidation_loss_proxy', 'ead'):.2%}")
print(f"EAD-weighted LGD after cure:  {exposure_weighted_average(result, 'lgd_after_cure_proxy', 'ead'):.2%}")

cure_stage_summary = (
    result.groupby('completion_stage', observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'count': len(g),
                'total_ead': g['ead'].sum(),
                'ead_weighted_cure_rate': exposure_weighted_average(g, 'cure_rate_proxy', 'ead'),
                'ead_weighted_lgd_after_cure': exposure_weighted_average(g, 'lgd_after_cure_proxy', 'ead'),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
display(cure_stage_summary.round(4))


## 9. APRA Slotting Framework

APRA may classify development finance as **specialised lending**, requiring a slotting approach:

| Slot | Risk Weight |
|------|-------------|
| Strong | 70% |
| Good | 90% |
| Satisfactory | 115% |
| Weak | 250% |

In [ ]:
slotted = engine.assign_slotting(result)

rows = []
for cat, grp in slotted.groupby('slotting_category', observed=True):
    rows.append({
        'slotting_category': cat,
        'count': len(grp),
        'total_ead': grp['ead'].sum(),
        'ead_weighted_lgd': exposure_weighted_average(grp, 'lgd_final', 'ead'),
        'mean_rw': grp['slotting_rw'].mean(),
        'mean_completion': grp['completion_pct'].mean(),
        'mean_presales': grp['presale_coverage'].mean(),
    })
slot_summary = pd.DataFrame(rows).set_index('slotting_category').round(4)

print('=== APRA Slotting Summary ===')
display(slot_summary)

# Illustrative slotted RWA
slotted['slotted_rwa'] = slotted['ead'] * slotted['slotting_rw']
print()
print(f"Total Slotted RWA: ${slotted['slotted_rwa'].sum():,.0f}")
print(f"Average Risk Weight: {slotted['slotted_rwa'].sum() / slotted['ead'].sum():.2%}")


In [ ]:
# Slotting distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

slot_counts = slotted['slotting_category'].value_counts().reindex(['Strong', 'Good', 'Satisfactory', 'Weak'])
slot_colours = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
axes[0].bar(slot_counts.index, slot_counts.values, color=slot_colours, edgecolor='white')
axes[0].set_title('Slotting Category Distribution')
axes[0].set_ylabel('Number of Loans')

slotted.boxplot(column='lgd_final', by='slotting_category', ax=axes[1])
axes[1].set_title('Final LGD by Slotting Category')
axes[1].set_ylabel('Final LGD')
plt.suptitle('')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'development_slotting.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Validation

In [ ]:
val_report = generate_validation_report(
    result,
    actual_col='realised_lgd',
    predicted_col='lgd_final',
    segment_col='completion_stage',
    date_col='default_date'
)

print("=== Accuracy ===")
for k, v in val_report['accuracy'].items():
    print(f"  {k}: {v}")

print("\n=== Conservatism Test ===")
for k, v in val_report['conservatism'].items():
    print(f"  {k}: {v}")

print("\n=== Calibration by Completion Stage ===")
display(val_report.get('calibration', 'N/A'))

In [ ]:
# Save outputs
slotted.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'development_lgd_results.csv'), index=False)
lr_lgd.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'development_segment_summary.csv'), index=False)
scenario_df.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'development_scenario_analysis.csv'), index=False)

print("Outputs saved to outputs/tables/")

---

## APS 113 Calibration Layer — Development Finance

> **SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY**
>
> This section adds a full APS 113-aligned calibration loop on top of the
> proxy baseline above. Workout data is synthetically generated (2014-2024,
> 10-year window) to demonstrate methodology; real production calibration
> requires an internal workout tape.

### Calibration Pipeline (APS 113 s.32-68)

| Step | Function | APS 113 |
|------|----------|---------|
| 1 | Load/generate historical workout data | s.44, Att A |
| 2 | `compute_realised_lgd()` — LIP costs, cure leg | s.32-34 |
| 3 | `classify_economic_regime()` + `assign_regime_to_workouts()` | s.43, s.46 |
| 4 | `segment_lgd()` — product-specific segment keys | s.52 |
| 5 | `compute_long_run_lgd()` — vintage-EWA method | s.43-44 |
| 6 | `compare_model_vs_actual()` — proxy vs realised | s.60-62 |
| 7 | `apply_downturn_overlay()` + `apply_correlation_adjustment()` | s.46-50, s.55-57 |
| 8 | `MoCRegister` + `apply_moc()` — 5 APS 113 s.65 sources | s.63-65 |
| 9 | `apply_regulatory_floor()` — 25-40% depending on stage per APS 113 s.58 | s.58 |
| 10 | Export 9 CSV outputs | — |
| 11 | `run_full_validation_suite()` — Gini, HL, PSI, OOT | s.66-68 |

**Correct APS 113 order:** LR-LGD → downturn overlay → correlation adj →
MoC → floor (MoC applied to downturn LGD, not long-run LGD, per s.63).


In [ ]:
# APS 113 Calibration Layer — imports and configuration
import os, sys
from pathlib import Path
sys.path.insert(0, os.path.abspath('..'))

from src.calibration_utils import (
    compute_realised_lgd,
    segment_lgd,
    compute_long_run_lgd,
    compare_model_vs_actual,
    classify_economic_regime,
    assign_regime_to_workouts,
    apply_downturn_overlay,
    apply_correlation_adjustment,
    build_lgd_pd_annual_series,
    estimate_lgd_pd_correlation,
    MoCRegister,
    apply_moc,
    apply_regulatory_floor,
    run_full_validation_suite,
    generate_compliance_map,
    export_compliance_map,
    CALIBRATION_STEP_ORDER,
)
from src.generators import GENERATOR_MAP, generate_all_historical_workouts

PRODUCT = "development_finance"
SEGMENT_KEYS = ['completion_stage_at_default', 'presale_cover_band']
MODEL_LGD_COL = "lgd_final"

HISTORY_DIR = Path('..') / 'data' / 'generated' / 'historical'
OUTPUTS_DIR = Path('..') / 'outputs' / 'tables'
UPSTREAM_PARQUET = Path('..') / 'data' / 'exports' / 'macro_regime_flags.parquet'

HISTORY_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Product: {PRODUCT}")
print(f"Segment keys: {SEGMENT_KEYS}")
print(f"APS 113 calibration pipeline — step order:")
for step, fn, ref in CALIBRATION_STEP_ORDER:
    print(f"  Step {step:>2}: {fn:<35} {ref}")


In [ ]:
# ── STEP 1: Load or generate historical workout data (APS 113 s.44, Att A) ─
parquet_path = HISTORY_DIR / f"{PRODUCT}_workouts.parquet"

if parquet_path.exists():
    cal_loans = pd.read_parquet(parquet_path)
    # cashflows stored alongside or re-generated
    try:
        cal_cashflows = pd.read_parquet(
            HISTORY_DIR / f"{PRODUCT}_cashflows.parquet"
        )
    except FileNotFoundError:
        cal_cashflows = None
    print(f"Loaded {len(cal_loans):,} historical workout loans from Parquet")
else:
    print(f"Parquet not found — generating synthetic workout data for {PRODUCT}")
    results = generate_all_historical_workouts(
        seed=42, output_dir=HISTORY_DIR, write_parquet=True, products=[PRODUCT]
    )
    cal_loans = results[PRODUCT]["loans"]
    cal_cashflows = results[PRODUCT].get("cashflows")
    print(f"Generated {len(cal_loans):,} synthetic workout loans (2014-2024)")

print(f"Date range: {cal_loans['default_date'].min()} to {cal_loans['default_date'].max()}")
print(f"Columns: {list(cal_loans.columns)}")

# ── STEP 2: Compute realised LGD (APS 113 s.32-34) ─────────────────────────
# LIP costs (Loss Identification Period, ~90 days) auto-detected if cashflows available
lgd_df = compute_realised_lgd(
    loans=cal_loans,
    cashflows=cal_cashflows,
    ead_col="ead_at_default",
    recovery_col="recovery_amount",
    cost_col="direct_costs",
    lip_window_days=90,
)
print(f"\nStep 2: Realised LGD computed")
print(f"  EAD-weighted realised LGD: {(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum():.2%}")
display(lgd_df[['realised_lgd', 'ead_at_default']].describe().round(4))

# ── STEP 3: Economic regime classification (APS 113 s.43, s.46-50) ─────────
upstream_path = str(UPSTREAM_PARQUET) if UPSTREAM_PARQUET.exists() else None
regime_df = classify_economic_regime(
    upstream_parquet_path=upstream_path,
    method="upstream_first",
)
print(f"\nStep 3: Economic regimes classified")
print(f"  Data source: {regime_df['data_source'].iloc[0]}")
display(regime_df[['year', 'regime', 'is_downturn_period', 'data_source']].head(12))

lgd_df = assign_regime_to_workouts(lgd_df, regime_df)
downturn_pct = lgd_df.get('is_downturn_period', pd.Series([False])).mean()
print(f"  Downturn observations: {downturn_pct:.1%} of portfolio")

# ── STEP 4: Segment by product-specific keys (APS 113 s.52) ────────────────
segmented_df = segment_lgd(lgd_df, SEGMENT_KEYS)
low_count = segmented_df[segmented_df.get('segment_flag', '') == 'low_count'] if 'segment_flag' in segmented_df.columns else pd.DataFrame()
print(f"\nStep 4: Segmentation complete")
print(f"  Segments: {segmented_df.groupby(SEGMENT_KEYS, observed=True).ngroups}")
if not low_count.empty:
    print(f"  Low-count segments flagged (<20 obs): {len(low_count)}")

# ── STEP 5: Long-run LGD — vintage-EWA (APS 113 s.43-44) ─────────────────
lr_lgd_df = compute_long_run_lgd(
    segmented_df,
    segment_keys=SEGMENT_KEYS,
    method="vintage_ewa",
)
print(f"\nStep 5: Long-run LGD by segment (vintage-EWA)")
display(lr_lgd_df.round(4))
lr_lgd_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_long_run_lgd_by_segment.csv", index=False)

# ── STEP 6: Compare model vs actual (APS 113 s.60-62) ──────────────────────
# 'model_lgd' here = proxy model LGD from the section above (lgd_final if present)
if MODEL_LGD_COL in cal_loans.columns:
    compare_input = lgd_df.merge(
        cal_loans[['loan_id', MODEL_LGD_COL] if 'loan_id' in cal_loans.columns else [MODEL_LGD_COL]],
        left_index=True, right_index=True, how='left',
    ) if MODEL_LGD_COL not in lgd_df.columns else lgd_df
else:
    compare_input = lgd_df.copy()
    compare_input['model_lgd'] = lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else 0.25

model_col_to_use = MODEL_LGD_COL if MODEL_LGD_COL in compare_input.columns else 'model_lgd'
mva_df = compare_model_vs_actual(
    compare_input,
    model_lgd_col=model_col_to_use,
    segment_keys=SEGMENT_KEYS,
)
print(f"\nStep 6: Model vs actual comparison")
display(mva_df.round(4))
mva_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_model_vs_actual_comparison.csv", index=False)

# ── STEP 7: Downturn overlay + Frye-Jacobs correlation adj (s.46-50, s.55-57)
# Reuses apply_downturn_overlay from existing lgd_calculation.py
dt_lgd = apply_downturn_overlay(segmented_df, product=PRODUCT)
print(f"\nStep 7: Downturn overlay applied")
if 'lgd_downturn' in dt_lgd.columns:
    ewa_dt = (dt_lgd['lgd_downturn'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    ewa_lr = (dt_lgd['realised_lgd'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    print(f"  EWA Long-run LGD:  {ewa_lr:.2%}")
    print(f"  EWA Downturn LGD:  {ewa_dt:.2%}")
    downturn_col = 'lgd_downturn'
else:
    dt_lgd['lgd_downturn'] = dt_lgd['realised_lgd'] * 1.15  # fallback scalar
    downturn_col = 'lgd_downturn'

# Frye-Jacobs correlation adjustment (APS 113 s.55-57)
try:
    lgd_ts, pd_ts = build_lgd_pd_annual_series(dt_lgd)
    macro_for_corr = regime_df.rename(columns={'gdp_growth': 'gdp_growth_yoy'})
    corr_result = estimate_lgd_pd_correlation(lgd_ts, pd_ts, macro_for_corr)
    dt_lgd['lgd_downturn_corr_adj'] = apply_correlation_adjustment(
        dt_lgd[downturn_col], corr_result['rho'], corr_result['macro_shock_std']
    )
    downturn_col = 'lgd_downturn_corr_adj'
    print(f"  Frye-Jacobs rho={corr_result['rho']:.3f}, adj factor={corr_result['lgd_dt_adjustment_factor']:.3f}")
except Exception as exc:
    print(f"  Frye-Jacobs skipped: {exc}")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_downturn_lgd_by_segment.csv", index=False)

# ── STEP 8: MoC register + apply MoC (AFTER downturn — APS 113 s.63-65) ───
# Determine regime data source for MoC model_approximation component
data_src = regime_df['data_source'].iloc[0] if 'data_source' in regime_df.columns else 'synthetic'
n_downturn_yrs = int(regime_df['is_downturn_period'].sum()) if 'is_downturn_period' in regime_df.columns else 0

psi_approx = 0.05  # placeholder — full PSI computed in Step 11
bias_approx = float(mva_df['bias'].abs().mean()) if 'bias' in mva_df.columns else 0.02

moc_register = MoCRegister(product=PRODUCT, regime_data_source=data_src)
moc_df = moc_register.build_moc_register(
    segment_df=segmented_df,
    segment_keys=SEGMENT_KEYS,
    n_downturn_vintages=n_downturn_yrs,
    psi_value=psi_approx,
    backtesting_bias=bias_approx,
)
print(f"\nStep 8: MoC register built")
display(moc_df.round(4))
moc_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_moc_register.csv", index=False)

lgd_with_moc = apply_moc(dt_lgd[downturn_col], moc_df, segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None)
dt_lgd['lgd_with_moc'] = lgd_with_moc
moc_ewa = (lgd_with_moc * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
print(f"  EWA LGD after MoC: {moc_ewa:.2%}")

# ── STEP 9: Regulatory floors (APS 113 s.58) ──────────────────────────────
dt_lgd['lgd_final_calibrated'] = apply_regulatory_floor(dt_lgd['lgd_with_moc'], product=PRODUCT)
final_ewa = (dt_lgd['lgd_final_calibrated'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
floor_binding_pct = (dt_lgd['lgd_final_calibrated'] > dt_lgd['lgd_with_moc']).mean()
print(f"\nStep 9: Regulatory floor applied")
print(f"  EWA Final Calibrated LGD: {final_ewa:.2%}")
print(f"  Floor binding for: {floor_binding_pct:.1%} of loans")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_final_calibrated_lgd.csv", index=False)

# ── STEP 10: Export all outputs ────────────────────────────────────────────
# Already exported: long_run_lgd_by_segment, model_vs_actual, downturn_lgd, moc_register, final_calibrated_lgd
# Export remaining:
lgd_df[['realised_lgd', 'ead_at_default']].assign(product=PRODUCT).to_csv(
    OUTPUTS_DIR / f"{PRODUCT}_historical_workouts.csv", index=False
)
regime_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_regime_classification.csv", index=False)

# Calibration adjustments summary
cal_adj_summary = pd.DataFrame({
    'product': [PRODUCT],
    'ewa_realised_lgd': [(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum()],
    'ewa_long_run_lgd': [lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None],
    'ewa_downturn_lgd': [(dt_lgd.get('lgd_downturn', dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_with_moc': [(dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_final': [final_ewa],
    'floor_binding_pct': [floor_binding_pct],
    'regime_data_source': [data_src],
    'n_loans': [len(lgd_df)],
    'calibration_date': [pd.Timestamp.today().date()],
})
cal_adj_summary.to_csv(OUTPUTS_DIR / f"{PRODUCT}_calibration_adjustments.csv", index=False)
print(f"\nStep 10: All outputs exported to {OUTPUTS_DIR}")

# ── STEP 11: Full validation suite (APS 113 s.66-68) ──────────────────────
print(f"\nStep 11: Running full validation suite...")
try:
    val_results = run_full_validation_suite(
        loans=dt_lgd,
        predicted_col='lgd_final_calibrated',
        actual_col='realised_lgd',
        segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None,
        date_col='default_date' if 'default_date' in dt_lgd.columns else None,
        product=PRODUCT,
    )
    print(f"  Gini: {val_results.get('gini', 'n/a')}")
    print(f"  Calibration ratio: {val_results.get('calibration_ratio', 'n/a')}")
    print(f"  PSI: {val_results.get('psi', 'n/a')}")
    if 'summary_table' in val_results:
        val_results['summary_table'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_validation_report.csv", index=False
        )
    if 'backtest_results' in val_results:
        val_results['backtest_results'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_backtest_results.csv", index=False
        )
except Exception as exc:
    print(f"  Validation suite error (non-fatal): {exc}")


In [ ]:
# ── Calibration summary waterfall ──────────────────────────────────────────
import matplotlib.pyplot as plt

try:
    stages = {
        'Realised LGD\n(2014-2024)': (lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum(),
        'Long-run LGD\n(vintage-EWA)': lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None,
        'Downturn LGD': (dt_lgd.get(downturn_col, dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        '+ MoC': (dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        'Final\n(Floor Applied)': final_ewa,
    }
    labels = [k for k, v in stages.items() if v is not None]
    values = [v for v in stages.values() if v is not None]
    colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#8e44ad'][:len(labels)]

    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.6)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.set_ylabel('EAD-Weighted LGD')
    ax.set_title(f'APS 113 Calibration Waterfall — Development Finance')
    ax.set_ylim(0, max(values) * 1.35)
    ax.axhline(values[-1], color='black', ls=':', lw=1, label=f'Final: {values[-1]:.1%}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        Path('..') / 'outputs' / 'figures' / f'development_finance_calibration_waterfall.png',
        dpi=150, bbox_inches='tight',
    )
    plt.show()
except Exception as exc:
    print(f"Waterfall chart error (non-fatal): {exc}")

# ── APS 113 compliance snapshot ─────────────────────────────────────────────
compliance_df = generate_compliance_map(
    calibration_results={PRODUCT: {'long_run_lgd_by_segment': True, 'calibration_steps': True}},
    moc_registers={PRODUCT: moc_df},
    regime_data_source=data_src,
    products=[PRODUCT],
)
print("\n=== APS 113 Compliance Snapshot ===")
display(compliance_df[['section_ref', 'requirement', 'status', 'reviewer_note']].set_index('section_ref'))
export_compliance_map(compliance_df, OUTPUTS_DIR / f"development_finance_aps113_compliance.csv")

# Final summary
print("\n=== Calibration Summary ===")
display(cal_adj_summary.round(4))
print(f"\nAll calibration outputs in: {OUTPUTS_DIR}")
print(f"SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY")
